<h1>Chapter 5 | Data Exercise #3 | <code>hotels-europe</code> | Generalizing from Data</h1>
<h2>Introduction:</h2>
<p>In this notebook, you will find my notes and code for Chapter 5's <b>exercise 3</b> of the book <a href="https://gabors-data-analysis.com/">Data Analysis for Business, Economics, and Policy</a>, by Gábor Békés and Gábor Kézdi. The question was: 
<p>3. Use <code>hotels-europe</code> dataset and pick two cities and the same date.
<p>Assignments:</p>
<ul>
    <li>In each city, take hotels with three stars and calculate the average price.</li>
    <li>Estimate the standard error of the estimated average price by bootsrap and using the SE formula.</li>
    <li>Create 95% confidence intervals.</li>
    <li>Compare the average price and the confidence intervals across the two cities, and explain why you have a narrower CI for one city than the other..</li>


</ul>
<h2>1. Load the data</h2>

In [2]:
import os
import pandas as pd
import warnings
from datetime import datetime
from plotnine import *
import sys
import numpy as np
from scipy.stats import norm, sem


warnings.filterwarnings("ignore")


In [3]:
# Increase number of returned rows in pandas
pd.set_option("display.max_rows", 500)

In [7]:
# Current script folder
dirname = os.getcwd()

# Get location folders
data_in = f"{dirname}/da_data_repo/hotels-europe/clean/"
data_out = f"{dirname}/da_data_exercises/ch05-generalizing_from_data/03-hotels_europe/data/clean/"
output = f"{dirname}/da_data_exercises/ch05-generalizing_from_data/03-hotels_europe/data/output/"
func = f"{dirname}/da_case_studies/ch00-tech_prep/"
sys.path.append(func)
paths = [data_in, data_out, output]

for path in paths:
    if not os.path.exists(path):
        os.makedirs(path)

In [8]:
# Import the prewritten helper functions 
from py_helper_functions import *

# Get the data
First, let's take a look at the data.

In [9]:
data_europe = pd.read_csv(f"{data_in}hotels-europe_price.csv")

In [10]:
data_europe.head()

,hotel_id,price,offer,offer_cat,year,month,weekend,holiday,nnights,scarce_room
0,1,172,0,0% no offer,2017,11,1,0,1,0
1,1,122,1,15-50% offer,2018,1,1,0,1,0
2,1,122,1,15-50% offer,2017,12,0,1,1,0
3,1,552,1,1-15% offer,2017,12,0,1,4,0
4,1,122,1,15-50% offer,2018,2,1,0,1,0


We probably need to do some data manipulation to get our features, which probably are at `hotels-europe_features.csv`.

In [11]:
hotels_europe_features = pd.read_csv(f"{data_in}hotels-europe_features.csv")

In [12]:
hotels_europe_features

,hotel_id,city,distance,stars,rating,country,city_actual,rating_reviewcount,center1label,center2label,neighbourhood,ratingta,ratingta_count,distance_alter,accommodation_type
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.0,Amsterdam,1.5,4.0,4.1,Netherlands,Amsterdam,165.0,City centre,Montelbaanstoren,Amsterdam,4.0,674.0,1.4,Hotel
2,4.0,Amsterdam,1.9,3.0,3.5,Netherlands,Amsterdam,298.0,City centre,Montelbaanstoren,Amsterdam,3.5,1882.0,2.1,Hotel
3,5.0,Amsterdam,1.8,3.5,4.0,Netherlands,Amsterdam,4.0,City centre,Montelbaanstoren,Amsterdam,4.5,66.0,2.0,Hotel
4,6.0,Amsterdam,1.9,4.0,4.1,Netherlands,Amsterdam,310.0,City centre,Montelbaanstoren,Amsterdam,4.0,767.0,2.0,Hotel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22897,19109.0,Rome,0.4,NaN,3.9,Italy,Rome,68.0,City centre,Grotta del Bue Marino,Trevi Fountain,4.0,100.0,0.7,Bed and breakfast
22898,19114.0,Rome,0.4,4.0,4.3,Italy,Rome,168.0,City centre,Palazzo Madama,Trevi Fountain,4.0,212.0,0.7,Hotel
22899,19115.0,Rome,0.4,NaN,4.3,Italy,Rome,44.0,City centre,Grotta del Bue Marino,Trevi Fountain,4.0,512.0,0.7,Guest House
22900,1.0,Amsterdam,3.1,4.0,4.3,Netherlands,Amsterdam,1030.0,City centre,Montelbaanstoren,Amsterdam,4.0,1115.0,3.6,Hotel


Ok, we'll work on this table first, select two cities and then join it with our facts table. We'll use Amsterdam and Madrid.

In [14]:
# Method 1: Your current approach (works fine, but not the most performant)
selected_cities = ["Amsterdam", "Madrid"]
hotels_cut_direct = hotels_europe_features[hotels_europe_features["city"].isin(selected_cities)]

In [16]:
hotels_cut_direct["city"].unique()

array(['Amsterdam', 'Madrid'], dtype=object)

In [17]:
hotels_cut_direct.head()

,hotel_id,city,distance,stars,rating,country,city_actual,rating_reviewcount,center1label,center2label,neighbourhood,ratingta,ratingta_count,distance_alter,accommodation_type
1,3.0,Amsterdam,1.5,4.0,4.1,Netherlands,Amsterdam,165.0,City centre,Montelbaanstoren,Amsterdam,4.0,674.0,1.4,Hotel
2,4.0,Amsterdam,1.9,3.0,3.5,Netherlands,Amsterdam,298.0,City centre,Montelbaanstoren,Amsterdam,3.5,1882.0,2.1,Hotel
3,5.0,Amsterdam,1.8,3.5,4.0,Netherlands,Amsterdam,4.0,City centre,Montelbaanstoren,Amsterdam,4.5,66.0,2.0,Hotel
4,6.0,Amsterdam,1.9,4.0,4.1,Netherlands,Amsterdam,310.0,City centre,Montelbaanstoren,Amsterdam,4.0,767.0,2.0,Hotel
5,7.0,Amsterdam,0.8,3.5,4.4,Netherlands,Amsterdam,258.0,City centre,Montelbaanstoren,Amsterdam,4.5,273.0,1.2,Hotel


In [18]:
# Multi-column filtering - several approaches

# Method 1: Chain boolean conditions with & (most common and readable)
hotels_filtered = hotels_europe_features[
    (hotels_europe_features["city"].isin(selected_cities)) &
    (hotels_europe_features["accommodation_type"] == "Hotel") &
    (hotels_europe_features["stars"] == 3.0)
]

print("Method 1 - Chained conditions:")
print(f"Shape: {hotels_filtered.shape}")
print(f"Cities: {hotels_filtered['city'].unique()}")
print(f"Accommodation types: {hotels_filtered['accommodation_type'].unique()}")
print(f"Stars: {hotels_filtered['stars'].unique()}")
print()

# Method 2: Using query method (very readable for complex conditions)
hotels_filtered_query = hotels_europe_features.query(
    "city in @selected_cities and accommodation_type == 'Hotel' and stars == 3.0"
)

print("Method 2 - Query method:")
print(f"Shape: {hotels_filtered_query.shape}")
print(f"Results are equal: {hotels_filtered.equals(hotels_filtered_query)}")
print()

# Method 3: Step-by-step filtering (good for debugging)
step1 = hotels_europe_features[hotels_europe_features["city"].isin(selected_cities)]
step2 = step1[step1["accommodation_type"] == "Hotel"]
hotels_filtered_steps = step2[step2["stars"] == 3.0]

print("Method 3 - Step-by-step:")
print(f"Shape: {hotels_filtered_steps.shape}")
print(f"Results are equal: {hotels_filtered.equals(hotels_filtered_steps)}")
print()

# Method 4: Using multiple conditions in a single boolean array
conditions = (
    hotels_europe_features["city"].isin(selected_cities) &
    (hotels_europe_features["accommodation_type"] == "Hotel") &
    (hotels_europe_features["stars"] == 3.0)
)
hotels_filtered_conditions = hotels_europe_features[conditions]

print("Method 4 - Separate conditions variable:")
print(f"Shape: {hotels_filtered_conditions.shape}")
print(f"Results are equal: {hotels_filtered.equals(hotels_filtered_conditions)}")

Method 1 - Chained conditions:
Shape: (211, 15)
Cities: ['Amsterdam' 'Madrid']
Accommodation types: ['Hotel']
Stars: [3.]

Method 2 - Query method:
Shape: (211, 15)
Results are equal: True

Method 3 - Step-by-step:
Shape: (211, 15)
Results are equal: True

Method 4 - Separate conditions variable:
Shape: (211, 15)
Results are equal: True


## Multi-column filtering best practices:

**Performance tips:**
1. **Use `&` and `|` instead of `and` and `or`** for boolean operations
2. **Put most selective conditions first** to reduce data early
3. **Use parentheses** around each condition to avoid operator precedence issues
4. **For complex logic, query() method** can be more readable

**Common gotchas:**
- Remember to use `&` (not `and`) and `|` (not `or`) for boolean arrays
- Always wrap individual conditions in parentheses when using `&` or `|`
- Use `==` for exact matches, `.isin()` for membership testing
- Be careful with NaN values - they might not match as expected

In [19]:
# Recommended approach for your analysis
# Clean, readable, and performant
hotels_cut = hotels_europe_features[
    (hotels_europe_features["city"].isin(selected_cities)) &
    (hotels_europe_features["accommodation_type"] == "Hotel") &
    (hotels_europe_features["stars"] == 3.0)
]

print("Final filtered dataset for analysis:")
print(f"Total hotels: {len(hotels_cut)}")
print("\nBreakdown by city:")
print(hotels_cut.groupby('city').size())

# Let's also check what columns we have available for the next steps
print(f"\nColumns available: {list(hotels_cut.columns)}")

Final filtered dataset for analysis:
Total hotels: 211

Breakdown by city:
city
Amsterdam    119
Madrid        92
dtype: int64

Columns available: ['hotel_id', 'city', 'distance', 'stars', 'rating', 'country', 'city_actual', 'rating_reviewcount', 'center1label', 'center2label', 'neighbourhood', 'ratingta', 'ratingta_count', 'distance_alter', 'accommodation_type']
